In [0]:
%pip install fitparse

In [0]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from scipy.signal import savgol_filter
import os
import glob

from utils.triathlon_utils import (
    parse_fit, 
    clean_fit_data, 
    calculate_elapsed_time, 
    apply_triathlon_style,
    format_time_axis,
    export_for_instagram,
    KLEUREN_PALETTE
)

from utils.config import DEFAULT_ATHLETE, DEFAULT_PATHS

In [0]:
def create_grade_impact_chart(df_stage, stage_num=9, garmin_hr=None, garmin_ascent=None, garmin_descent=None):
    """
    Genereert een visualisatie voor Etappe 9 inclusief statistiekenblok linksboven.
    Je kunt optioneel de officiële Garmin-waarden meegeven (garmin_hr, garmin_ascent, garmin_descent).
    """

    df = df_stage.copy()

    plt.style.use('dark_background')
    fig, ax = plt.subplots(figsize=(10, 6), facecolor='black')
    ax.set_facecolor('black')
    
    COLOR_MAGENTA = '#FF2A6D'
    
    # 1. Bereken stijgingspercentage en fallback hoogteverschillen
    if 'enhanced_altitude' in df.columns and 'distance' in df.columns:
        delta_alt = df['enhanced_altitude'].diff()
        delta_dist = df['distance'].diff()
        
        df['calculated_grade'] = np.where(
            (delta_dist > 0), 
            (delta_alt / delta_dist) * 100, 
            0
        )
        df['calculated_grade'] = df['calculated_grade'].fillna(0)
        
        calc_ascent = delta_alt[delta_alt > 0].sum()
        calc_descent = abs(delta_alt[delta_alt < 0].sum())
    else:
        df['calculated_grade'] = 0
        calc_ascent = 0
        calc_descent = 0

    hr_col = 'heart_rate' if 'heart_rate' in df.columns else None
    
    # Bepaal de waarden: gebruik de meegeleverde Garmin-data indien aanwezig, anders de berekening
    avg_hr = garmin_hr if garmin_hr is not None else (int(df[hr_col].mean()) if hr_col and not df[hr_col].dropna().empty else 0)
    total_ascent = garmin_ascent if garmin_ascent is not None else int(calc_ascent)
    total_descent = garmin_descent if garmin_descent is not None else int(calc_descent)
    
    if hr_col:
        # 2. Maak hellingsklassen (bins van 5%) om data te aggregeren
        bins = np.arange(-35, 40, 5)
        df['grade_bin'] = pd.cut(df['calculated_grade'], bins=bins)
        
        # Bereken gemiddelde en standaarddeviatie per klasse
        bin_stats = df.groupby('grade_bin', observed=False)[hr_col].agg(['mean', 'std', 'count']).reset_index()
        bin_stats['midpoint'] = bin_stats['grade_bin'].apply(lambda x: x.mid)
        bin_stats = bin_stats.dropna(subset=['mean'])
        
        # 3. Plot alle ruwe punten subtiel op de achtergrond
        ax.scatter(df['calculated_grade'], df[hr_col], c='gray', alpha=0.1, s=10, edgecolors='none')
        
        # 4. Plot de hoofdtrendlijn (gemiddelde hartslag per hellingscategorie)
        ax.plot(bin_stats['midpoint'], bin_stats['mean'], color=COLOR_MAGENTA, linewidth=3, marker='o', label='Gemiddelde hartslag')
        
        # 5. Voeg een vulgebied toe voor de spreiding (± 1 standaarddeviatie)
        ax.fill_between(
            bin_stats['midpoint'], 
            bin_stats['mean'] - bin_stats['std'], 
            bin_stats['mean'] + bin_stats['std'], 
            color=COLOR_MAGENTA, alpha=0.2, label='Spreiding (± 1 SD)'
        )
    
    ax.set_title(f"ETAPPE PERFORMANCE: IMPACT TERREIN OP HARTSLAG", color='white', fontsize=11, fontweight='bold', pad=12)
    ax.set_xlabel('Dalingspercentage / Stijging (%)', color='white', fontsize=10, fontweight='bold')
    ax.set_ylabel('Hartslag (bpm)', color=COLOR_MAGENTA, fontsize=10, fontweight='bold')
    
    ax.tick_params(colors='white', labelsize=9)
    ax.grid(True, linestyle='--', alpha=0.2, color='#374151')
    ax.set_xlim(-35, 35)
    
    # 6. Statistiekenblok toevoegen linksboven met jouw Garmin-waarden
    stats_text = (
        f"Gem. Hartslag: {avg_hr} bpm\n"
        f"Totale Stijging: +{total_ascent} m\n"
        f"Totale Daling: -{total_descent} m"
    )
    
    props = dict(boxstyle='round,pad=0.5', facecolor='#111827', edgecolor='#374151', alpha=0.85)
    ax.text(
        0.02, 0.95, stats_text, 
        transform=ax.transAxes, 
        fontsize=10, color='white', fontweight='bold',
        verticalalignment='top',
        bbox=dict(boxstyle='round,pad=0.5', facecolor='#111827', alpha=0.85, edgecolor='#374151')
    )

    # Legenda rechtsonder
    ax.legend(loc='lower right', facecolor='#111111', edgecolor='#374151', labelcolor='white', fontsize=9)
    
    plt.tight_layout()
    return fig

In [0]:
# Het FIT-bestand van etappe 8
fit_file_path = f"{DEFAULT_PATHS.fit_dir}/20260920_stubaier_hohenweg_etappe_9.fit"

# 1. Inlezen & Opschonen via de utils
df = parse_fit(fit_file_path)
df_clean = clean_fit_data(df)

# 2. Grafiek genereren
fig = create_grade_impact_chart(df_clean, stage_num=9, garmin_hr=127, garmin_ascent=899, garmin_descent=962)

# 3. Exporteren voor Instagram 
export_for_instagram(fig_stage9, filename="20260920_descent_impact.png", format_type='landscape')